state----agents status with respect to the environment 
action---- the decision taken in between the environment 
transition-- moving one state to another 
policy--- the rule of decision in an state 




creating states 

In [11]:
states=["s1","s2","s3","s4","s5","s6","s7","s8","s9"]
actions={"a1":"up","a2":"right","a3":"down","a4":"left","a5":"stay"}
grid ={
    "s1":(0,0),
    "s2":(0,1),
    "s3":(0,2),
    "s4":(1,0),
    "s5":(1,1),
    "s6":(1,2),
    "s7":(2,0),
    "s8":(2,1),
    "s9":(2,2)
}
position_to_state = {v:k for k,v in grid.items()}

In [14]:
def next_state(state,action):
    row,col=grid[state]
    if action == "a1":  # up
        row -= 1 
    elif action == "a2":  # right
        col += 1
    elif action == "a3":  # down
        row += 1
    elif action == "a4":  # left
        col -= 1
    elif action == "a5":  # stay
        pass
    
    
    if row < 0 or row > 2 or col < 0 or col > 2:
        return state  
    return position_to_state[(row, col)]

test the output 

In [15]:
print(next_state("s5","a1"))

s2


In [16]:
print(next_state("s2","a1"))

s2


deterministic policy 

In [18]:
policy = {
    "s1":"a2",
    "s2":"a2",
    "s3":"a3",
    "s4":"a1",
    "s5":"a1",
    "s6":"a4",
    "s7":"a5",
    "s8":"a5",
    "s9":"a5"}

In [19]:
def episode(start_state,steps=10):
    state = start_state
    trajectory=[state]
    
    for _ in range(steps):
        action = policy[state]
        state = next_state(state,action)
        trajectory.append(state)
        if state == "s9":
            break
    return trajectory

In [21]:
print(episode("s1"))

['s1', 's2', 's3', 's6', 's5', 's2', 's3', 's6', 's5', 's2', 's3']


In [22]:
print(episode("s4"))

['s4', 's1', 's2', 's3', 's6', 's5', 's2', 's3', 's6', 's5', 's2']


rewards 

In [23]:
def reward(state):
    if state == "s9":
        return 1
    else:
        return 0

In [24]:
def run_episode(start_state,steps=10):
    state = start_state
    trajectory=[state]
    total_reward=0
    
    for _ in range(steps):
        action = policy[state]
        state = next_state(state,action)
        trajectory.append(state)
        total_reward += reward(state)
        if state == "s9":
            break
    return trajectory,total_reward

In [25]:
trajectory,total_reward=run_episode("s1")
for step in trajectory:
    print(step)
print("Total Reward:", total_reward)

s1
s2
s3
s6
s5
s2
s3
s6
s5
s2
s3
Total Reward: 0


discounted return


In [30]:
def discounted_reward(trajectory,gamma=0.9):
   G = 0
   for t, state in enumerate(trajectory):
       G += (gamma**t) * reward(state)
   return G

In [33]:
trajectory = ["s1", "s2", "s3", "s6", "s9"]
G = discounted_reward(trajectory, gamma=0.9)
print("Discounted Reward:", G)

Discounted Reward: 0.6561


stochastic policy run 

In [34]:
stochastic_policy = {
    "s1": {"a2": 0.7, "a3": 0.3},
    "s2": {"a3": 0.8, "a2": 0.2},
    "s3": {"a3": 1.0},

    "s4": {"a2": 0.6, "a3": 0.4},
    "s5": {"a3": 0.7, "a2": 0.3},
    "s6": {"a3": 1.0},

    "s7": {"a2": 1.0},
    "s8": {"a2": 1.0},
    "s9": {"a5": 1.0}
}

In [35]:
import random

def choose_action(state):
    action_probs = stochastic_policy[state]

    actions = list(action_probs.keys())
    probs = list(action_probs.values())

    return random.choices(actions, weights=probs, k=1)[0]

In [36]:
def run_stochastic_episode(start_state, steps=10):
    state = start_state
    trajectory = []
    total_reward = 0

    for _ in range(steps):
        action = choose_action(state)
        next_s = next_state(state, action)
        r = reward(next_s)

        trajectory.append((state, action, next_s, r))
        total_reward += r

        state = next_s

        if state == "s9":
            break

    return trajectory, total_reward

In [39]:
trajectory, total_reward = run_stochastic_episode("s1")

for step in trajectory:
    print(step)

print("Total reward:", total_reward)

discounted_return = sum((0.9 ** t) * step[3] for t, step in enumerate(trajectory))
print("Discounted return:", discounted_return)

('s1', 'a2', 's2', 0)
('s2', 'a2', 's3', 0)
('s3', 'a3', 's6', 0)
('s6', 'a3', 's9', 1)
Total reward: 1
Discounted return: 0.7290000000000001
